# 👩🏻‍🍳👩🏻‍💼👨🏻‍🔧 <span style="color: white; background-color: tomato"><b> Automação do relatório de Colaboradores </b></span></p>

🔧 <span style="color: chocolate"><b> 1- Extrai automaticamente a base de colaboradores (COLAB) </b></span></p>
- A automação abre o Datamace via Selenium + PyAutoGUI
- Navega até o módulo GIP
- Solicita e exporta o relatório de headcount
- Aguarda a conclusão e captura o arquivo gerado na pasta monitorada

🧹 <span style="color: chocolate"><b> 2- Processa e limpa a base </b></span></p>
- Utiliza o Pandas
- Lê o arquivo COLAB extraído
- Elimina duplicidades, ajusta colunas e renomeia variáveis
- Aplica ordenações e padronizações
- Gera uma base final padronizada e tratada

📝 <span style="color: chocolate"><b> 3- Gera uma planilha Excel formatada </b></span></p>
- Cria COLABORADORES.xlsx
- Constrói uma tabela formatada dentro da planilha, pronta para uso operacional e analítico

📊 <span style="color: chocolate"><b> 4- Atualiza o dashboard no Power BI </b></span></p>
- Abre o arquivo HEADCOUNT.pbix
- Atualiza as consultas
- Publica o relatório
- Fecha o aplicativo após o refresh

🔄 <span style="color: chocolate"><b> Atualiza a planilha corporativa de controle (Controle HC e Atestados) </b></span></p>
- Abre o arquivo .xlsx
- Navega para células específicas
- Dispara os comandos de refresh. 
- Garante que o relatório de controle esteja sincronizado

🗃 <span style="color: chocolate"><b> Gera a base em CSV para o banco de dados corporativo </b></span></p>
- tb_colaboradores.csv
- Pronto para ingestão no banco ou pipeline de dados

📝 <span style="color: chocolate"><b> Mantém um arquivo de log e auditoria onde registra </b></span></p>
- Cada etapa executada  
- Horários  
- Possíveis erros  
- Tempo total do processo
- Tudo em PROCESSOS.xlsx

# Importando as Bibliotecas

In [16]:
import logging
import os
import pandas as pd
import pyautogui
import pygetwindow as gw
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium.common.exceptions import WebDriverException
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [17]:
CONFIG = {
    'id_processo': 7,
    'caminho_registros': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'diretorio_extracao': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'diretorio_movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'arquivo_final': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
}

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Caminhos de pastas definidas')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Caminhos de pastas definidas

----------------------------------------------------------------------------------------------------


# Registra Etapa do Processo

In [18]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [19]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o GIP

In [20]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o GIP
time.sleep(15) # Tempo maior para aguardar carregar o GIP
pyautogui.click(x=168, y=185) # Maximiza a janela
time.sleep(2)
pyautogui.click(x=168, y=185) # Clica no GIP
time.sleep(5)
pyautogui.click(x=1079, y=240) # Fecha janela de Tarefas Anuais 
time.sleep(3)
pyautogui.click(x=1109, y=301) # Geradores
time.sleep(2)
pyautogui.click(x=1168, y=325) # Arq./Relatório 
time.sleep(5)
pyautogui.click(x=546, y=364) # Lupa
time.sleep(2)
pyautogui.press("enter")
time.sleep(2)
pyautogui.click(x=615, y=614) # Arquivo
time.sleep(2)
pyautogui.click(x=664, y=517) # Multi processamento
time.sleep(2)
pyautogui.click(x=876, y=544) # Estrutura
time.sleep(2)
pyautogui.click(x=831, y=630) # Excel
time.sleep(2)
pyautogui.click(x=685, y=681) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão da Base COLAB

In [21]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base COLAB?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-26 09:47:50,053 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-26 09:47:50,067 - INFO - Iniciando a leitura e processamento dos dados...


# Retorna Arquivos COLAB Presentes no Diretório

In [22]:
def obter_arquivos_colaboradores(diretorio):
    if not diretorio.exists():
        print(f"❌ Diretório não existe: {diretorio}")
        return []
    
    lista_arquivos = [f.name for f in diretorio.glob('COLAB*.xlsx')]
    return lista_arquivos

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLAB na pasta')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Arquivo COLAB na pasta

----------------------------------------------------------------------------------------------------


# Processa Arquivo de Colaboradores

In [23]:
def processar_colaboradores(arquivo):
    # 1. Leitura do arquivo
    df = pd.read_excel(arquivo)
    
    # 2. Criar coluna auxiliar de data para ordenação (equivalente ao CASE WHEN do SQL)
    if 'RESC_DAT' in df.columns:
        df['RESC_DAT_TEMP'] = pd.to_datetime(df['RESC_DAT'], errors='coerce')
    else:
        df['RESC_DAT_TEMP'] = pd.NaT
        
    # Preenche valores nulos com data futura para garantir que fiquem no topo/fundo da ordenação
    df['RESC_DAT_ORDENACAO'] = df['RESC_DAT_TEMP'].fillna(pd.Timestamp('2050-12-31'))
    
    # 3. Ordenação
    # Ordena por REGISTRO, depois SITUACAO (Crescente) e depois DATA DE RESCISÃO (Decrescente)
    df = df.sort_values(
        by=['REGISTRO', 'SITUACAO', 'RESC_DAT_ORDENACAO'],
        ascending=[True, True, False]
    )
    
    # 4. Remoção de duplicatas
    # Mantém apenas a primeira ocorrência de cada REGISTRO após a ordenação
    df_processado = df.drop_duplicates(subset=['REGISTRO'], keep='first')
    
    # 5. Dicionário de mapeamento
    colunas_mapeamento = {
        'REGISTRO': 'registro',
        'NOME_COMP': 'nome',
        'NASCTO': 'nascimento',
        'SEXO': 'sexo',
        'RACA_COR': 'etnia_raca',
        'FORMACAO': 'cod_formacao',
        'APOSENTADO': 'aposentado',
        'RG_NUMERO': 'rg', 
        'CPF_NUMERO': 'cpf',
        'DEFICIENTE': 'deficiente',
        'NACIONALI': 'nacionalidade',
        'NAT.UF_ORI': 'naturalidade_uf',
        'NAT_CIDADE': 'naturalidade_cidade',
        'EST_CIVIL': 'estado_civil',
        'SEXO_CONJU': 'sexo_conjuge',
        'NOME_CONJU': 'nome_conjuge',
        'DT_NAS_CON': 'data_nasc_conjuge',
        'CPF_CONJUG': 'cpf_conjuge',
        'FILHOS': 'filhos',
        'ADM_DATA': 'data_admissao',
        'SITU_DESCR': 'situacao',
        'RESC_DAT': 'data_rescisao',
        'RESC_DESCR': 'descricao_rescisao',
        'COD_EMPRES': 'cod_empresa',            
        'TAB_CARGO': 'cod_cargo',
        'CARGO_COMP': 'cargo_completo',
        'CARGO_DRES': 'cargo_abreviado',
        'TAB_CC_CON': 'cod_centro_custo',
        'NOM_GEST_C': 'nome_gestor',
        'TAB_SECAO': 'cod_secao',
        'SECAO': 'secao',
        'CCUSTO_CON': 'centro_custo',
        'SAL_ADMISS': 'salario_admissao',
        'SAL_ATUAL': 'salario_atual',
        'SAL_TOTAL': 'salario_total',
        'FGT_SALDO': 'saldo_fgts',
        'ENDERECO': 'endereco',
        'END_NUMERO': 'numero_endereco',
        'COMPLEMEN': 'complemento',
        'BAIRRO': 'bairro',
        'CIDADE': 'cidade',
        'ESTADO': 'estado',
        'CEP': 'cep',
        'TELEFONE1': 'telefone',
        'E_MAIL': 'email',
        'E_MAIL 2': 'segundo_email',
        'SITUACAO': 'cod_situacao',
        'REG_TRABAL': 'regime_trabalho',
        'HOR_BASE': 'hora_base',
        'HOR_COMPL.': 'hora_complemento',
        'PRO_DT_IND': 'ultimo_reajuste_individual',
        'PRO_DT_COL': 'ultimo_reajuste_coletivo',
        'FER_DT_INI': 'data_inicio_ferias',
        'FER_DT_FIM': 'data_fim_ferias'
    }
    
    # 6. Filtra apenas as colunas que realmente existem no DataFrame para evitar KeyError
    colunas_presentes = [col for col in colunas_mapeamento.keys() if col in df_processado.columns]
    
    # 7. Seleciona e renomeia as colunas finais
    df_final = df_processado[colunas_presentes].rename(columns=colunas_mapeamento)
    
    return df_final

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Tratamento da base COLAB finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Tratamento da base COLAB finalizado

----------------------------------------------------------------------------------------------------


# Cria Arquivo em Excel com Formatação de Tabela

In [24]:
def criar_arquivo_excel(df, caminho, nome_tabela="COLABORADORES"):

    wb = Workbook()
    ws = wb.active
    ws.title = nome_tabela
    
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)
    
    tabela = Table(displayName=nome_tabela, ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    wb.close()

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLABORADORES criado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Arquivo COLABORADORES criado

----------------------------------------------------------------------------------------------------


# Finalizando Processo

In [25]:
# Execução Principal
def main():
    config = CONFIG
    inicio = datetime.now()
    
    try:
        # Etapa 0
        append_registro_processo(config['caminho_registros'], config['id_processo'], 0)
        
        # Etapa 1 - Listar arquivos
        arquivos = obter_arquivos_colaboradores(config['diretorio_extracao'])
        
        if not arquivos:
            print("⚠️ Nenhum arquivo COLAB encontrado")
            return
        
        append_registro_processo(config['caminho_registros'], config['id_processo'], 1)
        
        # Processar arquivos
        for arquivo_nome in arquivos:
            try:
                arquivo_caminho = config['diretorio_extracao'] / arquivo_nome
                print(f"🔄 Processando: {arquivo_nome}")
                
                df_processado = processar_colaboradores(arquivo_caminho)
                criar_arquivo_excel(df_processado, config['arquivo_final'])
                
                # Mover arquivo
                destino = config['diretorio_movidos'] / arquivo_nome
                shutil.move(str(arquivo_caminho), str(destino))
                
                print(f"✅ {arquivo_nome} concluído")
                
            except Exception as e:
                print(f"❌ Erro ao processar {arquivo_nome}: {e}")
                continue
        
        # Etapa 8 - Finalização
        append_registro_processo(config['caminho_registros'], config['id_processo'], 2)
                
        duracao = datetime.now() - inicio
        
        print('—' * 100)
        print(f'\n   ✅ Processo finalizado com sucesso')
        print(f'   ⏱️  Tempo de execução: {duracao}\n')
        print('—' * 100)
        
    except Exception as e:
        print(f"❌ Erro crítico na execução: {e}")
        raise

if __name__ == "__main__":
    main()

🔄 Processando: COLAB___062026.XLSX


c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ COLAB___062026.XLSX concluído
————————————————————————————————————————————————————————————————————————————————————————————————————

   ✅ Processo finalizado com sucesso
   ⏱️  Tempo de execução: 0:00:16.660837

————————————————————————————————————————————————————————————————————————————————————————————————————


# Atualização do Power BI - HEADCOUNT

In [26]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\HEADCOUNT.pbix"
os.startfile(path_pbi) # Abre o arquivo 
time.sleep(40)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(3)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI Atualizado

----------------------------------------------------------------------------------------------------


# Atualizando o Arquivo Excel Controle HC e Atestados

In [27]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('HC!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'f5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [28]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_colaboradores.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [29]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:11:55.839263

----------------------------------------------------------------------------------------------------
